# Lot 45 — G5 : cibles MCTS32768 uniformes (G5_DEEP_AUTONOMOUS_REANALYSIS_V1)

Notebook **mince** : la logique vit dans `apps/trainer/scripts/lot45/` (+ infrastructure validée `lot44/`)
et le runner `run_lot45_g5_target_generation.py`. Seule la cellule **CONFIGURATION** se modifie.

| Étape | Effet | Bloquant |
|---|---|---|
| 1 Mount Drive | monte Drive, vérifie `songo-ai/` | oui |
| 2 Locate project | clone de la branche + extraction vérifiée de `lot45_inputs.tar.gz` | oui |
| 3 Environment | installation, CUDA | oui |
| 4 Preflight | `selftest` + `preflight` (dont RUN_LOCK) → **STOP si FAIL** | oui |
| 5 Smoke test | pipeline complet budget 64, format final du dataset | oui |
| 6 Prepare | validation Lot44, plan de shards pour `TARGET_POSITIONS`, `compute_plan.json` | oui |
| 7 Pilot | premiers shards réels MCTS32768 (réutilisés) → débit, mémoire, taille | oui |
| 8 Generate / Resume | calcul sous verrou mono-écrivain ; relancer = reprise | `CONFIRM_LONG_COMPUTE` |
| 9 Finalize | validation, dataset, audits, `decision.json` | oui |
| 10 Export | `lot45_results.tar.gz` vérifié | oui |

**Un seul écrivain** : si une autre machine (Mac, autre runtime) possède `run_lock.json` avec un heartbeat récent,
la génération refuse de démarrer. Aucun entraînement, aucun label Minimax/teacher, aucune position à 65536.

In [ ]:
# ===================== CONFIGURATION (seule cellule à modifier) =====================
REPO_URL = 'https://github.com/GlennEriss/songo-fish.git'
REPO_BRANCH = 'dev'
EXPECTED_COMMIT = ''           # recommandé : SHA affiché à l'étape 2, pour les reprises
DRIVE_ROOT_OVERRIDE = ''       # '' = /content/drive/MyDrive
PROJECT_ROOT_OVERRIDE = ''     # '' = /content/songo-fish
DEVICE = 'auto'                # 'auto' | 'cuda' | 'cpu'
SEED = 20264501
TARGET_POSITIONS = 10000       # préfixe de la sélection ordonnée (voir compute_plan.json)
CONCURRENCY = 128              # arbres MCTS32768 en parallèle (~6.4 Go RAM à 128)
MAX_SHARDS_PER_SESSION = None  # None = tout ; un entier = découpe la session
TAKEOVER_STALE_LOCK = False    # True uniquement si l'ancien écrivain est MORT (heartbeat expiré)
CONFIRM_LONG_COMPUTE = False   # True pour lancer l'étape 8
# ====================================================================================

## STEP 1 — Mount Drive

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, hashlib, tarfile

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path(DRIVE_ROOT_OVERRIDE or '/content/drive/MyDrive')
SONGO_AI = DRIVE_ROOT / 'songo-ai'
if not SONGO_AI.is_dir():
    raise RuntimeError(f'songo-ai introuvable sous {DRIVE_ROOT}')
INPUT_BUNDLE = SONGO_AI / 'inputs' / 'lot45_inputs.tar.gz'
print('DRIVE_ROOT =', DRIVE_ROOT, '| INPUT_BUNDLE', 'OK' if INPUT_BUNDLE.is_file() else 'MISSING')

## STEP 2 — Locate project

In [ ]:
REPO_DIR = Path(PROJECT_ROOT_OVERRIDE or '/content/songo-fish')
if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'merge', '--ff-only', f'origin/{REPO_BRANCH}'], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if EXPECTED_COMMIT and not COMMIT.startswith(EXPECTED_COMMIT):
    raise RuntimeError(f'commit {COMMIT} != EXPECTED_COMMIT {EXPECTED_COMMIT}')
RUNNER = REPO_DIR / 'apps/trainer/scripts/run_lot45_g5_target_generation.py'
if not RUNNER.is_file() or not (REPO_DIR / 'apps/trainer/scripts/lot45/generation.py').is_file():
    raise RuntimeError(f'pipeline Lot45 absent sur origin/{REPO_BRANCH} @ {COMMIT[:10]}')

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

if not INPUT_BUNDLE.is_file():
    raise RuntimeError(f'{INPUT_BUNDLE} absent : lancer en local `--stage prepare-inputs --sync-to-drive`')
if file_sha256(INPUT_BUNDLE) != Path(str(INPUT_BUNDLE) + '.sha256').read_text().split()[0]:
    raise RuntimeError('checksum du bundle inputs Lot45 invalide')
with tarfile.open(INPUT_BUNDLE, 'r:gz') as archive:
    archive.extractall(REPO_DIR)
manifest = json.loads((REPO_DIR / 'lot45_input_manifest.json').read_text())
bad = [f for f, digest in manifest['files'].items() if file_sha256(REPO_DIR / f) != digest]
if bad:
    raise RuntimeError(f'fichiers inputs corrompus : {bad}')
os.chdir(REPO_DIR)
print('REPO_DIR =', REPO_DIR, '| commit =', COMMIT, '| inputs vérifiés :', len(manifest['files']))

## STEP 3 — Environment

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[train,perf,dev]'], check=True)
import torch
print('torch', torch.__version__, '| CUDA_AVAILABLE =', 'YES' if torch.cuda.is_available() else 'NO', '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError("DEVICE='cuda' mais CUDA indisponible")

BASE = ['--drive-root', str(DRIVE_ROOT), '--device', DEVICE, '--seed', str(SEED), '--concurrency', str(CONCURRENCY)]
GENERATE = (['--max-shards', str(MAX_SHARDS_PER_SESSION)] if MAX_SHARDS_PER_SESSION else []) + (['--takeover-stale-lock'] if TAKEOVER_STALE_LOCK else [])
OUTPUT = SONGO_AI / 'experiments' / 'lot45_g5_uniform_mcts32768_targets'

def lot45(stage, *extra):
    """Lance un stage du runner en diffusant sa sortie ; lève une erreur si le stage échoue."""
    cmd = [sys.executable, '-u', str(RUNNER), '--stage', stage, *BASE, *extra]
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'Lot45 stage {stage} FAILED (code {proc.returncode}); voir {OUTPUT}/errors.jsonl')

print(json.dumps({'commit': COMMIT, 'device': DEVICE, 'target_positions': TARGET_POSITIONS, 'concurrency': CONCURRENCY, 'max_shards_per_session': MAX_SHARDS_PER_SESSION, 'takeover_stale_lock': TAKEOVER_STALE_LOCK, 'confirm_long_compute': CONFIRM_LONG_COMPUTE, 'output': str(OUTPUT)}, indent=2))

## STEP 4 — Preflight
**Le notebook s'arrête ici si PREFLIGHT_STATUS ≠ PASS.**

In [ ]:
lot45('selftest')
lot45('preflight')
PREFLIGHT = json.loads((OUTPUT / 'preflight_report.json').read_text())
for name, status in PREFLIGHT['summary'].items():
    print(f'{name:<24} {status}')
if PREFLIGHT['PREFLIGHT_STATUS'] != 'PASS':
    raise RuntimeError('PREFLIGHT FAIL — STOP')

## STEP 5 — Smoke test

In [ ]:
lot45('smoke')
if json.loads((OUTPUT / 'smoke_test_report.json').read_text())['SMOKE_TEST'] != 'PASS':
    raise RuntimeError('SMOKE FAIL — STOP')

## STEP 6 — Prepare
Le plan de shards est un **préfixe** de la sélection ordonnée : on peut ensuite l'étendre (10k → 25k) sans rien recalculer.

In [ ]:
lot45('prepare', '--target-positions', str(TARGET_POSITIONS))
PLAN = json.loads((OUTPUT / 'compute_plan.json').read_text())
for c in PLAN['candidates']:
    print(f"{c['positions']:>7} positions | {c['estimated_t4_hours']:6.1f} h T4 | {c['estimated_storage_mb']:6.0f} Mo | {c['shards']} shards")

## STEP 7 — Pilot
Premiers shards réels MCTS32768 sous le protocole final (réutilisés dans le dataset).

In [ ]:
lot45('pilot')
PILOT = json.loads((OUTPUT / 'pilot_report.json').read_text())
print({k: PILOT[k] for k in ('PILOT', 'positions', 'device', 'seconds_per_position', 'simulations_per_second', 'peak_rss_bytes', 'artifact_bytes_per_position')})
if PILOT['PILOT'] != 'PASS':
    raise RuntimeError('PILOT FAIL — STOP')
lot45('plan')

## STEP 8 — Generate / Resume
Relancer la cellule reprend au shard en cours. Si `RUN_LOCKED` : une autre machine écrit encore — l'arrêter d'abord.

In [ ]:
if not CONFIRM_LONG_COMPUTE:
    lot45('generate')  # affiche le plan sans calculer
    raise RuntimeError('CONFIRM_LONG_COMPUTE = False : relire le plan puis passer True dans la CONFIGURATION')
lot45('generate', '--confirm', *GENERATE)
lot45('status')

## STEP 9 — Finalize

In [ ]:
lot45('finalize')
print(json.dumps(json.loads((OUTPUT / 'decision.json').read_text()), indent=2))

## STEP 10 — Export

In [ ]:
EXPORT = SONGO_AI / 'exports' / 'lot45_results.tar.gz'
lot45('export', '--bundle', str(EXPORT))
print('EXPORT =', EXPORT)